## Observability with LangSmith
LangSmith records every model and tool call so you can see exactly what your graph did. Sign up at [smith.langchain.com](https://smith.langchain.com), create an API key, and add these lines to your `.env` file:
```text
LANGSMITH_TRACING=true
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=agentic-track
```
That is all. Every LangChain and LangGraph run is then traced automatically with no change to your code, and you can watch it unfold on the LangSmith dashboard. We will not run this here, but it is well worth turning on for your own work.

In [ ]:
import os
import random
import warnings
from typing import Annotated

warnings.filterwarnings("ignore", category=DeprecationWarning)

import gradio as gr
import requests
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict


load_dotenv(override=True)
NTFY_TOPIC_URL = os.getenv("NTFY_TOPIC_URL")

## Memory: why the graph forgets, and how to fix it
Try chatting twice with the graph above and it will not remember the first exchange. That feels odd, because the state was carefully passing messages along. The reason is that one call to `invoke` is one run of the graph: a sequence of super-steps, one for each layer of nodes that executes (the LLM call is one super-step; the tool calls it requests run together in the next). The reducer accumulates messages within that run, but the next `invoke` starts fresh.

To remember across calls we add a checkpointer. It saves a snapshot of the state after every super-step, filed under a `thread_id`. Give it the same `thread_id` next time and the saved history is loaded before your new message is added.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

memory = MemorySaver()

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))
builder.add_node("translator", translator_node)
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition, {"tools": "tools", END: "translator"})
builder.add_edge("translator", END)
builder.add_edge("tools", "chatbot")
graph = builder.compile(checkpointer=memory)


config = {"configurable": {"thread_id": "conversation-1"}}
graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Ed."}]}, config)
second = graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)
print(second["messages"][-1].content)
print(second["spanish"])